In [1]:

import numpy as np
import random

random.seed(42)
np.random.seed(42)

days = 100
initial_price = 100

daily_returns = np.random.normal(0.001, 0.02, days)
prices = initial_price * np.cumprod(1 + daily_returns)

POPULATION_SIZE = 20
PARENTS_COUNT = 4
GENERATIONS = 10
TRANSACTION_FEE = 0.001
DRAWDOWN_PENALTY = 0.5


def create_strategy():
    short = random.randint(2, 15)
    long = random.randint(16, 40)
    return [short, long]


def evaluate(strategy):
    short, long = strategy

    short_ma = np.convolve(
        prices, np.ones(short) / short, mode="full"
    )[:days]

    long_ma = np.convolve(
        prices, np.ones(long) / long, mode="full"
    )[:days]

    signal = (short_ma > long_ma).astype(float)

    position = np.roll(signal, 1)
    position[0] = 0

    trades = np.abs(np.diff(position, prepend=0))

    market_returns = np.zeros(days)
    market_returns[1:] = prices[1:] / prices[:-1] - 1

    strategy_returns = (
        position * market_returns - trades * TRANSACTION_FEE
    )

    portfolio = np.cumprod(1 + strategy_returns)

    net_profit = (portfolio[-1] - 1) * 100

    peak = np.maximum.accumulate(portfolio)
    drawdowns = (peak - portfolio) / peak
    max_drawdown = np.max(drawdowns) * 100

    total_fees = np.sum(trades * TRANSACTION_FEE) * 100

    fitness = net_profit - DRAWDOWN_PENALTY * max_drawdown

    return fitness, net_profit, max_drawdown, total_fees


def crossover(parent1, parent2):
    child = [
        random.choice([parent1[0], parent2[0]]),
        random.choice([parent1[1], parent2[1]])
    ]

    if child[0] >= child[1]:
        child[0], child[1] = 2, 20

    return child


def mutate(strategy):
    child = strategy.copy()

    if random.random() < 0.3:
        child[0] = random.randint(2, 15)

    if random.random() < 0.3:
        child[1] = random.randint(16, 40)

    return child


population = [
    create_strategy() for _ in range(POPULATION_SIZE)
]

best_strategy = None
best_fitness = -float("inf")

for generation in range(GENERATIONS):

    population.sort(
        key=lambda s: evaluate(s)[0],
        reverse=True
    )

    fitness, profit, drawdown, fees = evaluate(population[0])

    if fitness > best_fitness:
        best_fitness = fitness
        best_strategy = population[0].copy()

    print(
        f"Generation {generation + 1}: "
        f"Best Profit = {profit:.2f}%, "
        f"Drawdown = {drawdown:.2f}%"
    )

    parents = population[:PARENTS_COUNT]
    new_population = [p.copy() for p in parents]

    while len(new_population) < POPULATION_SIZE:
        parent1, parent2 = random.sample(parents, 2)
        child = crossover(parent1, parent2)
        child = mutate(child)
        new_population.append(child)

    population = new_population


fitness, profit, drawdown, fees = evaluate(best_strategy)

print("\n----- BEST TRADING STRATEGY -----")
print("Short Moving Average:", best_strategy[0])
print("Long Moving Average:", best_strategy[1])
print(f"Net Profit: {profit:.2f}%")
print(f"Maximum Drawdown: {drawdown:.2f}%")
print(f"Transaction Fees: {fees:.2f}% of initial capital")
print(f"Fitness Score: {fitness:.2f}")


Generation 1: Best Profit = 1.30%, Drawdown = 13.95%
Generation 2: Best Profit = 1.30%, Drawdown = 13.95%
Generation 3: Best Profit = 2.18%, Drawdown = 13.95%
Generation 4: Best Profit = 6.05%, Drawdown = 13.95%
Generation 5: Best Profit = 6.05%, Drawdown = 13.95%
Generation 6: Best Profit = 6.05%, Drawdown = 13.95%
Generation 7: Best Profit = 6.05%, Drawdown = 13.95%
Generation 8: Best Profit = 6.05%, Drawdown = 13.95%
Generation 9: Best Profit = 6.05%, Drawdown = 13.95%
Generation 10: Best Profit = 6.05%, Drawdown = 13.95%

----- BEST TRADING STRATEGY -----
Short Moving Average: 14
Long Moving Average: 20
Net Profit: 6.05%
Maximum Drawdown: 13.95%
Transaction Fees: 0.50% of initial capital
Fitness Score: -0.93
